## Build container using the lock file:

In [1]:
import modal
import os
from pathlib import Path
from importlib.metadata import distributions
from pydantic import BaseModel
from dataclasses import dataclass
from packaging.version import Version
import json
from datasets import Dataset, load_from_disk

In [2]:
@dataclass
class ServerConfig:
    packages: list[str]
    gpu_config: str

In [3]:
assert Version(modal.__version__) > Version("1.1"), "Modal > 1.1 is required"

In [4]:
root_dir = Path("../../").resolve()

In [5]:
## Replicate prepare_data.py to verify
from rich import print as rprint
from IPython.display import JSON



In [ ]:
app = modal.App("build-image-test")
image = (
    modal.Image.debian_slim(python_version="3.12")
    .uv_sync(
        uv_project_dir="../../", extra_options="--no-dev")
    .add_local_file(
        "../../scripts/train_baseline.py", 
        remote_path="/root/train_baseline.py")
)

data_vol = modal.Volume.from_name(
    name="n24-data", 
    create_if_missing=False)
runs_vol = modal.Volume.from_name(
    name="n24-runs",
    create_if_missing=True,
)

In [8]:
@app.function(image=image, gpu="L4")
def get_package_versions():
    from importlib.metadata import distributions
    import subprocess
    from dataclasses import dataclass

    @dataclass
    class ServerConfig:
        packages: list[str]
        gpu_config: str

    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    packages = sorted(
        f"{dist.metadata['Name']}=={dist.version}"
        for dist in distributions()
    )
    return ServerConfig(packages=packages, gpu_config=result.stdout.strip())


In [ ]:
@app.function(image=image, volumes={"/news24-data": data_vol})
def get_dataset_counts():
    SPLIT_FILES = (
    "train",
    "dev",
    "test",
    )
    prepared_data = Path("/news24-data/data/prepared")

    records = {
        key: len(Dataset.from_file(str(prepared_data / key / "data-00000-of-00001.arrow")))
        for key in SPLIT_FILES
    }
    
    return records


@app.function(image=image, volumes={"/news24-runs": runs_vol})
def write_simple_json():
    import json
    data = {"test": "test"}
    with open("/news24-runs/smoke-1.json", "w") as fp:
        json.dump(data, fp, indent=4)
    return None

In [ ]:
with app.run():
    result = get_dataset_counts.remote()
    write_simple_json.remote()
    ## result = get_package_versions.remote

In [19]:
from rich import print as rprint
rprint(result)

{'train': 48988, 'dev': 6123, 'test': 6124}